In [1]:
import pickle
import os
import urllib

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torchvision.transforms.functional as TF
from sklearn.decomposition import PCA
from scipy import signal

DINOV3_GITHUB_LOCATION = "facebookresearch/dinov3"

if os.getenv("DINOV3_LOCATION") is not None:
    DINOV3_LOCATION = os.getenv("DINOV3_LOCATION")
else:
    DINOV3_LOCATION = DINOV3_GITHUB_LOCATION

print(f"DINOv3 location set to {DINOV3_LOCATION}")

/opt/anaconda3/envs/kir_env/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/opt/anaconda3/envs/kir_env/lib/python3.11/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


DINOv3 location set to facebookresearch/dinov3


In [2]:
import torch

# Choose model type
MODEL_NAME = "dinov3_vitl16"

# Load model from GitHub (weights not needed, we'll load our checkpoint)
model = torch.hub.load(
    'facebookresearch/dinov3:main',  # GitHub repo
    MODEL_NAME,
    pretrained=False  # do NOT download weights
)

# Move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Load your local checkpoint
checkpoint_path = r"/home/kirtangangani/dinov3/checkpoint/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth"
state_dict = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()


Using cache found in /home/kirtangangani/.cache/torch/hub/facebookresearch_dinov3_main


DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-23): 24 x SelfAttentionBlock(
      (norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): LinearKMaskedBias(in_features=1024, out_features=3072, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
    )
  )
  (norm)

In [3]:
PATCH_SIZE = 16
IMAGE_SIZE = 768

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# image_uri = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQmFP2n_nogMiQm24-g53ON7yZq-KCjgnzLlw&s"

def load_image_from_url(url: str) -> Image:
    with urllib.request.urlopen(url) as f:
        return Image.open(f).convert("RGB")

# image resize transform to dimensions divisible by patch size
def resize_transform(
    mask_image: Image,
    image_size: int = IMAGE_SIZE,
    patch_size: int = PATCH_SIZE,
) -> torch.Tensor:
    w, h = mask_image.size
    h_patches = int(image_size / patch_size)
    w_patches = int((w * image_size) / (h * patch_size))
    return TF.to_tensor(TF.resize(mask_image, (h_patches * patch_size, w_patches * patch_size)))


# image = load_image_from_url(image_uri)
image = Image.open("/home/kirtangangani/dinov3/Thermal_Images/Thermal_1.JPG")
image_resized = resize_transform(image)
image_resized_norm = TF.normalize(image_resized, mean=IMAGENET_MEAN, std=IMAGENET_STD)

In [4]:
# DINOv3 model names
MODEL_DINOV3_VITS  = "dinov3_vits16"
MODEL_DINOV3_VITSP = "dinov3_vits16plus"
MODEL_DINOV3_VITB  = "dinov3_vitb16"
MODEL_DINOV3_VITL  = "dinov3_vitl16"
MODEL_DINOV3_VITHP = "dinov3_vith16plus"
MODEL_DINOV3_VIT7B = "dinov3_vit7b16"

# Select the model
MODEL_NAME = MODEL_DINOV3_VITL

# Map number of layers
MODEL_TO_NUM_LAYERS = {
    MODEL_DINOV3_VITS: 12,
    MODEL_DINOV3_VITSP: 12,
    MODEL_DINOV3_VITB: 12,
    MODEL_DINOV3_VITL: 24,
    MODEL_DINOV3_VITHP: 32,
    MODEL_DINOV3_VIT7B: 40,
}

n_layers = MODEL_TO_NUM_LAYERS[MODEL_NAME]


In [30]:
MODEL_TO_NUM_LAYERS = {
    MODEL_DINOV3_VITS: 12,
    MODEL_DINOV3_VITSP: 12,
    MODEL_DINOV3_VITB: 12,
    MODEL_DINOV3_VITL: 24,
    MODEL_DINOV3_VITHP: 32,
    MODEL_DINOV3_VIT7B: 40,
}

n_layers = MODEL_TO_NUM_LAYERS[MODEL_NAME]

with torch.inference_mode():
    with torch.autocast(device_type='cuda', dtype=torch.float32):
        # feats = model.get_intermediate_layers(image_resized_norm.unsqueeze(0).cuda(), n=range(n_layers), reshape=True, norm=True, return_class_token=False)
        feats = model.get_intermediate_layers(image_resized_norm.unsqueeze(0).cuda(), n=range(n_layers), reshape=True, norm=True, return_class_token=True)
        print(type(feats[-1]))
        print(len(feats[-1]))
        print(feats[-1][0].shape)
        print(feats[-1][1].shape)

        # x = feats[-1].squeeze().detach().cpu()
        # print(x.shape)
        # dim = x.shape[0]
        # x = x.view(dim, -1).permute(1, 0)

<class 'tuple'>
2
torch.Size([1, 1024, 48, 36])
torch.Size([1, 1024])


In [6]:
x.shape

torch.Size([1728, 1024])

In [ ]:
with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float32):
    feats = model.get_intermediate_layers(
        image_resized_norm.unsqueeze(0).cuda(),
        n=range(n_layers),
        reshape=True,
        norm=True,
        return_class_token=True
    )

print(type(feats))
patch_tokens, cls_token = feats[-1]

# Remove batch dimension
patch_tokens = patch_tokens.squeeze(0)
cls_token = cls_token.squeeze(0).squeeze(0)

print("Patch tokens:", patch_tokens.shape)
print("CLS token:", cls_token.shape)

<class 'tuple'>
Patch tokens: torch.Size([1024, 48, 36])
CLS token: torch.Size([1024])
